# Radiometry Estimates
Current problem is that my radiometry is code is not agreeing with my Curio graphs. I think I'll see what I can do here to figure out what's reasonable and what isn't.  

## By Hand Writing New Functions
Checking that the Mathemtica SNR agrees with a re-calculation by "first principles"

In [68]:
import math
from math import pi, sqrt
import plotly.express as px
import numpy as np # Used here to generate ordered data easily

In [59]:
def test():
    print('this is a string from a function defined right here in a Jupyter notebook!')
test()

this is a string from a function defined right here in a Jupyter notebook!


In [60]:
def mag(x):
    return(-2.5 * math.log10(x))
def amag(x):
    return(10**(-0.4 * x))

In [61]:
cal = 866000 # photons/cm^2/sec
back = 21.61 # Magnitudes per square arcsecond

In [62]:
def signal(x, aper, integrationTime, q):
    ''' x is the source in magnitudes
    aper is the aperture size in m
    IntegrationTime is in seconds, 
    q is the net quantum efficiency'''
    area = math.pi * (aper*100/2)**2
    return(  amag(x) * cal *  area * integrationTime * q  )

In [83]:
def noise(aper, photaper, integrationTime, q):
    '''
    aper is the aperture of the telescope in meters
    photaper is the size of the photometry aperture RADIUS in arcseconds (assumed to be square)
    integration time is in seconds
    calibration and background are picked up from the notebook
    q is the net quantum efficiency 
    '''
    area = math.pi * (aper*100/2)**2
    print('    aperture area', area)
    photarea = math.pi * (photaper/2)**2
    print('    photarea', photarea)
    return( math.sqrt(amag(back) * cal * area *integrationTime * q * photarea ))

In [85]:
print('signal', signal(20.5, 1, 10, 0.2))
print('noise', noise(1, 2, 10, 0.2))
print('snr', signal(20.5, 1, 10, 0.2)  / noise(1, 2, 10, 0.2)  )

signal 85.82973448778654
    aperture area 7853.981633974483
    photarea 3.141592653589793
noise 9.849038176984214
    aperture area 7853.981633974483
    photarea 3.141592653589793
snr 8.714529575929381


This section basically did an recalculation from first principles of the SNR to compare withe the value we seem to have in the Mathematica code.  Turns out to be 8.7 vs 10. for the SNR,  but the value of 20.5 was just eyeballed from the graph, and the calibraiton is from a different source and in different units, so I think good agreement.

## Checking the equation in the Equations Document 

Here I'm calculating the required integration time using  3 from the Equations document.  It should end up being about 10 if I have the numbers right in the Mathematica notebook as used above.  Really this is just a check that equation 3 is derived correcdtly.

In [93]:
pi = math.pi
radperster = 2 * pi / (3600 * 360)
print(radperster)

4.84813681109536e-06


Matching the parameters used in the last section, although converted to m^2.

In [104]:
gamma = 8.71 # This is the SNR we got in the last section
A = math.pi * (100/2)**2
beta = amag(back) * cal / (radperster**2)
omega = pi * (radperster**2)
alpha = amag(20.5) * cal 
eta = 0.2
f = 1
print('gamma', gamma)
print('A', A)
print('beta', beta)
print('omega', omega)
print('alpha', alpha)
print('eta', eta)
print('f', f)
print( 'Integration Time' , ( (gamma**2) * beta * omega) /( (alpha**2) * A * eta * f**2))

gamma 8.71
A 7853.981633974483
beta 83631167.2585127
omega 7.38413463084415e-11
alpha 0.00546409060319846
eta 0.2
f 1
Integration Time 9.989607244511332


This does compare with the last section.

In [87]:
print(alpha * A * 10 * eta)

85.82973448778654


In [91]:
print(sqrt(beta * A * 10 * 0.2 * eta * omega))

4.404623775345462


## Using the Code

In [42]:
print(signal(20.5, 1, 10, 0.2))
print(noise(1, 1, 10, 0.2))
print(signal(20.5, 1, 10, 0.2)  / noise(1, 2, 10, 0.2)  )

858297.3448778654
492.4519088492107
871.4529575929382


In [43]:
print(noise(1, 1, 10, 0.2))

492.4519088492107


In [44]:
print(signal(20.5, 1, 10, 0.2)  / noise(1, 2, 10, 0.2)  )

871.4529575929382


## Discussion
This result is pretty much with the Mathematica code gives, seeing as I'm reading things off the figure by eye. I note that I was using the photomotery RADIUS instead of the DIAMETER in mathematica, somewhat unexpected. 
## Conclusion
My old mathematica calculation and this much rougher calculation agree.  Now what's going on with my python code?

# Python Code Testing
Most of this is in detector.py which relies on constants.py and rediometry_data.py

In [110]:
from importlib import reload
import detector
reload(detector)

<module 'detector' from '/Users/espillar/Desktop/vibevolts/detector.py'>

In [111]:
detector.testdetector()

gamma 10
beta [6.49888933e+11]
omega [7.38413463e-11]
alpha [55.46115058]
A [0.78539816]
eta [0.2]
f [1.]
[9.93210085]


#### This should be 10 seconds, not 2.48 second.  Hmfff.

In [ ]:
gamma 8.71
A 7853.981633974483
beta 83631167.2585127
omega 7.38413463084415e-11
alpha 0.00546409060319846
eta 0.2
f 1
Integra

## Comparing:
- gamma: 10 vs. 8.71 - that's set , OK,
- $\eta$, $f$, $A$ are OK
- $\omega$ is OK
- $\alpha$ is off by $\sim 10^4$ in the python code
- $\beta$ is off by $\sim 10^4$ in the python code actuall 7771

In [105]:
6.49888933e+11 / 83631167.2585127

7770.893965776242

In [106]:
0.00546409060319846 / 55.46115058

9.852104664357399e-05

In [107]:
83631167.2585127 / 6.49888933e+11

0.00012868532300197317

In [108]:
1.84603366e-11 / 7.38413463084415e-11

0.25000000030998387

In [112]:
55.46115058 / 0.00546409060319846

10150.115473476091

In [114]:
print(6.49888933e+11 /83631167.2585127)

7770.893965776242
